# 🐍 Master Guide: Multithreading, GIL & Thread Safety
> **Single-Source Revision Guide**: Covers the Global Interpreter Lock (GIL), OS thread mechanics, race conditions, and synchronization primitives (`Lock`, `RLock`, `Semaphore`).
>
> *Run each cell interactively with `Shift + Enter`.*

---

## 📌 1. Topic Overview
Python multithreading allows concurrent execution of multiple threads sharing the same memory space. While ideal for **I/O-bound** operations (network requests, disk I/O), the Global Interpreter Lock (GIL) restricts execution of Python bytecode to one thread at a time.

---
## 🧠 2. CPython Architecture: The Global Interpreter Lock (GIL)

### Why Does the GIL Exist?
CPython's memory management uses **reference counting** (`ob_refcnt`). Without the GIL, concurrent access to reference counts from multiple OS threads would cause race conditions, memory leaks, and fatal crashes.

```
CPU Core 1 ───┐
CPU Core 2 ───┼──> [ Global Interpreter Lock (GIL) ] ──> PyEval_EvalFrameDefault()
CPU Core 3 ───┘
               Only ONE thread executes Python bytecode at any moment!
```

- **I/O-Bound Tasks**: Threads **release the GIL** while waiting for network/disk responses. Multithreading provides massive speedups for I/O!
- **CPU-Bound Tasks**: Threads fight over the GIL. Multithreading will NOT speed up CPU-bound tasks in standard CPython (use `multiprocessing` instead).

---
## 🟢 3. Part 1: Beginner Fundamentals

### Step 1: Spawning Threads with `threading.Thread`

In [ ]:
import threading
import time

def download_file(filename, delay):
    print(f"[Thread {threading.current_thread().name}] Starting download: {filename}...")
    time.sleep(delay)  # Releases the GIL!
    print(f"[Thread {threading.current_thread().name}] Finished: {filename}")

t1 = threading.Thread(target=download_file, args=("data_1.csv", 0.2), name="Worker-1")
t2 = threading.Thread(target=download_file, args=("data_2.csv", 0.2), name="Worker-2")

t1.start()
t2.start()

t1.join()  # Wait for t1 to finish
t2.join()  # Wait for t2 to finish
print("All downloads completed!")

---
## 🟡 4. Part 2: Intermediate Features

### Race Conditions & `threading.Lock`
When multiple threads modify shared mutable state, thread switching can interrupt non-atomic operations (like `counter += 1`), causing race conditions.

In [ ]:
shared_counter = 0
lock = threading.Lock()

def safe_increment():
    global shared_counter
    for _ in range(10_000):
        with lock:  # Acquires lock and releases deterministically
            shared_counter += 1

threads = [threading.Thread(target=safe_increment) for _ in range(5)]
for t in threads: t.start()
for t in threads: t.join()

print("Final counter (Expected 50,000):", shared_counter)

---
## 🔴 5. Part 3: Tricky Parts, Gotchas & Interview Traps

### ⚠️ Gotcha 1: Deadlocks & Re-entrant Locks (`RLock`)
A standard `Lock` cannot be acquired twice by the same thread! Trying to re-acquire it causes a permanent **deadlock**. Use `threading.RLock()` if a thread needs to re-enter a locked method.

In [ ]:
rlock = threading.RLock()

with rlock:
    print("Acquired outer RLock")
    with rlock:  # Safe! RLock knows the calling thread already holds it
        print("Acquired nested RLock safely")

---
## 📋 6. Part 4: Quick Revision Cheat Sheet

```
+------------------------------------+---------------------------------------+
| Concept                            | Key Rule                              |
+------------------------------------+---------------------------------------+
| GIL                                | One thread executes bytecode at a time|
| Best for                           | I/O-bound tasks (network, disk, DB)   |
| Worst for                          | CPU-bound tasks (use multiprocessing) |
| threading.Lock                     | Prevents race conditions on shared var|
| threading.RLock                    | Re-entrant lock for recursive calls   |
| t.join()                           | Blocks calling thread until t finishes|
+------------------------------------+---------------------------------------+
```

---
## 🎓 Practical Hands-On & Real-World Examples

### 1. The 2 Approaches for Developing Thread-Based Applications (`NumGenFunEx1.py`, `NumGenOOPEx1.py`)
> **Core Multi-Threading Architecture**:
> - **Approach 1 (Functional)**: Create an instance of `threading.Thread(target=function_name, name="Worker-1", args=(...))` and invoke `.start()`.
> - **Approach 2 (OOP Subclassing)**: Inherit from `threading.Thread`, override the `run()` method, instantiate the subclass, and invoke `.start()`.
>
> ⚠️ **Important API Note**: `getName()` and `setName()` are deprecated. Use the modern `.name` attribute.

In [ ]:
# The 2 Thread Creation Approaches Drill
import threading
import time

print(f"Active Main Thread: '{threading.current_thread().name}' | Total Active: {threading.active_count()}")

# Approach 1: Functional Target
def task_worker(worker_id, count):
    t_name = threading.current_thread().name
    for i in range(1, count + 1):
        print(f"  [{t_name}] Step {i}/{count}")
        time.sleep(0.01)

# Approach 2: OOP Subclassing threading.Thread
class OOPThreadWorker(threading.Thread):
    def __init__(self, name, iterations):
        super().__init__(name=name)
        self.iterations = iterations

    def run(self):
        for i in range(1, self.iterations + 1):
            print(f"  [{self.name}] OOP Execution Step {i}")
            time.sleep(0.01)

t1 = threading.Thread(target=task_worker, name="Func-Worker", args=(1, 3))
t2 = OOPThreadWorker(name="OOP-Worker", iterations=3)

t1.start()
t2.start()
t1.join()
t2.join()
print("Both thread approaches completed successfully!")

### 2. Thread Synchronization & Critical Sections with `threading.Lock` (`SyncBankTransOOPsEx1.py`, `SyncReservationOOPEx1.py`)
Preventing race conditions during simultaneous bank withdrawals or ticket reservations.

In [ ]:
# Practical Implementation: SyncBankTransOOPsEx1.py
# Race condition prevention using threading.Lock

class BankAccountSimulation:
    def __init__(self, initial_balance=5000):
        self.balance = initial_balance
        self.lock = threading.Lock()

    def withdraw(self, customer_name, amount):
        print(f"[{customer_name}] Attempting to withdraw Rs. {amount}...")
        
        # Critical Section protected by Lock
        with self.lock:
            print(f"  [{customer_name}] Entered Critical Section (Lock Acquired)")
            if self.balance >= amount:
                time.sleep(0.02)  # Simulating processing latency
                self.balance -= amount
                print(f"  [{customer_name}] Withdrawal SUCCESS! Remaining Balance: Rs. {self.balance}")
            else:
                print(f"  [{customer_name}] Withdrawal FAILED: Insufficient Balance (Available: Rs. {self.balance})")
            print(f"  [{customer_name}] Exiting Critical Section (Lock Released)\n")

bank = BankAccountSimulation(initial_balance=5000)

# Simulate 3 simultaneous ATM withdrawal requests
threads = [
    threading.Thread(target=bank.withdraw, args=("Customer-A (Aditya)", 3000)),
    threading.Thread(target=bank.withdraw, args=("Customer-B (Bhavna)", 3000)),
    threading.Thread(target=bank.withdraw, args=("Customer-C (Chaitra)", 1000))
]

for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"Final Account Balance: Rs. {bank.balance}")